# Backfill Generated Eval For Old Submissions

Put the old `submission.zip` files under the Drive input folders, then run the cells in order. This notebook contains the backfill code directly; it does not need a separate `.py` file.


In [ ]:
# Run this first in a fresh GPU Colab runtime.
# If this import fails, restart/factory-reset the runtime before doing anything else.
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("This is not a GPU runtime. Use Runtime > Change runtime type > GPU, then restart.")
print("gpu:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Use A100/H100 or another BF16-capable GPU for Nemotron backfill.")

# Keep setup minimal. Do not install torch here.
!pip install -q packaging ninja bitsandbytes accelerate peft transformers safetensors
!pip install -q mamba-ssm --no-build-isolation
!pip uninstall -y -q causal-conv1d

import mamba_ssm
print("mamba_ssm import ok")

from google.colab import drive
from pathlib import Path
drive.mount("/content/drive", force_remount=False)

INPUT_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/Kaggle challenges/nemotron_challenge/artefacts/eval_backfill/input")
(INPUT_ROOT / "00-raw-1024").mkdir(parents=True, exist_ok=True)
(INPUT_ROOT / "02-raw-full").mkdir(parents=True, exist_ok=True)
print("Put 0.62 submission here:", INPUT_ROOT / "00-raw-1024" / "submission.zip")
print("Put 0.54 submission here:", INPUT_ROOT / "02-raw-full" / "submission.zip")
print("If needed, put competition data zip here:", INPUT_ROOT / "nvidia-nemotron-model-reasoning-challenge.zip")
!find "/content/drive/MyDrive/Colab_Notebooks/Kaggle challenges/nemotron_challenge/artefacts/eval_backfill/input" -maxdepth 3 -type f -print


In [ ]:
"""Backfill generated-eval CSVs for an already submitted adapter.

Run this in Colab after installing the same Nemotron dependencies as the
training notebook. It does not train. It loads saved submission.zip files,
scores the fixed 256-row eval split plus fixed probe rows, and writes for each run:

- generated_eval_predictions.csv
- generated_eval_summary.csv
- probe_evolution.csv

Copy each run's two files into the matching local submission archive folder
and rerun scripts/build_experiment_dashboard.py.
"""

from __future__ import annotations

import json
import re
import time
import types
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd


PROJECT_ROOT = Path("/content/nemotron_challenge")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_ZIP_NAME = "nvidia-nemotron-model-reasoning-challenge.zip"
DATA_ZIP_PATH = Path("/content") / DATA_ZIP_NAME
DRIVE_ARTEFACTS_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/Kaggle challenges/"
    "nemotron_challenge/artefacts"
)
BACKFILL_INPUT_DIR = DRIVE_ARTEFACTS_ROOT / "eval_backfill" / "input"
BACKFILL_OUTPUT_DIR = DRIVE_ARTEFACTS_ROOT / "eval_backfill" / "outputs"

SYSTEM_PROMPT = (
    "Solve the puzzle and output only the final answer value. Do not explain. "
    "Do not write prefixes like 'answer:' or 'the answer is'."
)
EVAL_ROWS = 256
PROBE_ROWS = 5
MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 64
GENERATION_BATCH_SIZE = 2  # Increase to 4 only if Colab has clear VRAM headroom.
SUMMARY_STEP = 0
PRECISION_MODE = "bf16"


@dataclass(frozen=True)
class BackfillRun:
    label: str
    output_dir: Path
    candidate_zip_paths: tuple[Path, ...]
    max_new_tokens: int = MAX_NEW_TOKENS
    step: int = 0


BACKFILL_RUNS = [
    BackfillRun(
        label="00-raw-1024",
        output_dir=BACKFILL_OUTPUT_DIR / "00-raw-1024",
        candidate_zip_paths=(
            BACKFILL_INPUT_DIR / "00-raw-1024" / "submission.zip",
            BACKFILL_INPUT_DIR / "2026-05-16_colab_nemotron_lora_score_0_62" / "submission.zip",
            BACKFILL_INPUT_DIR / "00-raw-1024_submission.zip",
        ),
    ),
    BackfillRun(
        label="02-raw-full",
        output_dir=BACKFILL_OUTPUT_DIR / "02-raw-full",
        candidate_zip_paths=(
            BACKFILL_INPUT_DIR / "02-raw-full" / "submission.zip",
            BACKFILL_INPUT_DIR / "2026-05-17_colab_raw_full_r4_score_0_54" / "submission.zip",
            BACKFILL_INPUT_DIR / "02-raw-full_submission.zip",
        ),
    ),
]


def infer_family(prompt: str) -> str:
    text = str(prompt).lower()
    if "bit manipulation" in text or "8-bit binary" in text:
        return "bit_manipulation"
    if "secret encryption" in text or "decrypt" in text:
        return "cipher"
    if "numeral system" in text:
        return "numeral"
    if "unit" in text and "convert" in text:
        return "unit_conversion"
    if "gravitational constant" in text or "falling distance" in text:
        return "gravity"
    if "transformation rules is applied to equations" in text or "determine the result for" in text:
        return "equation"
    return "unknown"


def normalize_answer(value: Any) -> str:
    text = "" if value is None else str(value).strip()
    text = re.sub(r"^`+|`+$", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip().strip(".").lower()


def extract_final_answer(text: str) -> str:
    text = "" if text is None else str(text)
    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    if boxed_starts:
        matches = []
        for idx, match in enumerate(boxed_starts):
            start = match.end()
            end = boxed_starts[idx + 1].start() if idx + 1 < len(boxed_starts) else len(text)
            segment = text[start:end]
            last_brace = segment.rfind("}")
            matches.append(segment[:last_brace] if last_brace != -1 else segment)
        non_empty = [match.strip() for match in matches if match.strip()]
        return normalize_answer(non_empty[-1] if non_empty else matches[-1])

    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:\uff1a]\s*([^\n]+)",
        r"final answer\s*[:\uff1a]\s*([^\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return normalize_answer(matches[-1])

    number_matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if number_matches:
        return normalize_answer(number_matches[-1])

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        line = re.sub(r"^(final answer|answer|assistant)\s*[:\-]\s*", "", line, flags=re.I).strip()
        if line:
            return normalize_answer(line.strip("`").strip())
    return normalize_answer(text)


def build_prompt(prompt: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{prompt}\n\nAssistant:\n"


def find_data_zip() -> Path:
    candidates = [
        DATA_ZIP_PATH,
        BACKFILL_INPUT_DIR / DATA_ZIP_NAME,
        DRIVE_ARTEFACTS_ROOT.parent / DATA_ZIP_NAME,
    ]
    for path in candidates:
        if path.exists():
            return path
    candidate_text = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"No competition data zip found. Put it in one of:\n{candidate_text}")


def load_rows() -> tuple[pd.DataFrame, pd.DataFrame]:
    if not (RAW_DIR / "train.csv").exists():
        RAW_DIR.mkdir(parents=True, exist_ok=True)
        data_zip_path = find_data_zip()
        print("using data zip:", data_zip_path)
        with zipfile.ZipFile(data_zip_path) as archive:
            archive.extractall(RAW_DIR)
    train = pd.read_csv(RAW_DIR / "train.csv")
    rows = train[["id", "prompt", "answer"]].copy()
    rows["family"] = rows["prompt"].map(infer_family)
    eval_rows = rows.sample(n=min(EVAL_ROWS, len(rows) - 1), random_state=42).reset_index(drop=True)
    probe_rows = rows.groupby("family", group_keys=False).head(1).head(PROBE_ROWS).reset_index(drop=True)
    return eval_rows, probe_rows


def find_submission_zip(run: BackfillRun) -> Path:
    for path in run.candidate_zip_paths:
        if path.exists():
            return path
    candidates = "\n".join(f"- {path}" for path in run.candidate_zip_paths)
    raise FileNotFoundError(
        f"No submission.zip found for {run.label}. Put it in one of:\n{candidates}"
    )


def extract_adapter(run: BackfillRun) -> Path:
    adapter_dir = run.output_dir / "adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    submission_zip_path = find_submission_zip(run)
    print("using submission zip:", submission_zip_path)
    with zipfile.ZipFile(submission_zip_path) as archive:
        archive.extractall(adapter_dir)
    return adapter_dir


def patch_nemotron_moe_dtype(current_model, torch_module) -> None:
    def _patched_nemotron_moe(self, hidden_states, topk_indices, topk_weights):
        final_hidden_states = torch_module.zeros_like(hidden_states, dtype=topk_weights.dtype)
        expert_mask = torch_module.nn.functional.one_hot(topk_indices, num_classes=len(self.experts))
        expert_mask = expert_mask.permute(2, 0, 1)
        for expert_idx in range(len(self.experts)):
            expert = self.experts[expert_idx]
            mask = expert_mask[expert_idx]
            token_indices, weight_indices = torch_module.where(mask)
            if token_indices.numel() > 0:
                expert_weights = topk_weights[token_indices, weight_indices]
                expert_input = hidden_states[token_indices]
                expert_output = expert(expert_input)
                weighted_output = expert_output * expert_weights.unsqueeze(-1)
                final_hidden_states.index_add_(0, token_indices, weighted_output.to(final_hidden_states.dtype))
            else:
                expert_dtype = expert.down_proj.weight.dtype
                dummy_input = torch_module.zeros_like(hidden_states[0]).unsqueeze(0).to(expert_dtype)
                final_hidden_states = final_hidden_states + expert(dummy_input).to(final_hidden_states.dtype)
        return final_hidden_states.to(hidden_states.dtype)

    patched = 0
    for module in current_model.modules():
        if module.__class__.__name__ == "NemotronHMOE":
            module.moe = types.MethodType(_patched_nemotron_moe, module)
            patched += 1
    print("patched Nemotron MoE dtype modules:", patched)


def map_adapter_key(key: str) -> str:
    mapped = key
    for suffix in ["lora_A", "lora_B", "lora_embedding_A", "lora_embedding_B"]:
        mapped = mapped.replace(f".{suffix}.weight", f".{suffix}.default.weight")
    return mapped


def load_adapter_weights_directly(model, adapter_dir: Path) -> None:
    from safetensors.torch import load_file

    adapter_path = adapter_dir / "adapter_model.safetensors"
    adapter_state = load_file(str(adapter_path))
    model_state = model.state_dict()
    mapped_state = {}
    missing = []
    for key, tensor in adapter_state.items():
        mapped_key = map_adapter_key(key)
        candidates = [mapped_key]
        if not mapped_key.startswith("base_model.model."):
            candidates.append("base_model.model." + mapped_key)
        target_key = next((candidate for candidate in candidates if candidate in model_state), None)
        if target_key is None:
            missing.append(key)
        else:
            mapped_state[target_key] = tensor.to(model_state[target_key].dtype)

    if missing:
        preview = "\n".join(missing[:20])
        raise KeyError(f"Could not map {len(missing)} adapter weights. First missing keys:\n{preview}")
    result = model.load_state_dict(mapped_state, strict=False)
    print(f"loaded {len(mapped_state)} adapter tensors directly from {adapter_path}")
    if result.unexpected_keys:
        print("unexpected keys:", result.unexpected_keys[:10])


def load_model(adapter_dir: Path):
    import torch
    from google.colab import userdata
    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    adapter_config = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))
    model_name = adapter_config["base_model_name_or_path"]
    hf_token = userdata.get("hftoken")

    if PRECISION_MODE != "bf16":
        raise ValueError("This backfill script currently expects PRECISION_MODE='bf16'.")
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("Use an A100/H100 Colab runtime for this Nemotron eval.")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, token=hf_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        torch_dtype=torch.bfloat16,
        device_map={"": 0},
        trust_remote_code=True,
        token=hf_token,
    )
    base_model.config.use_cache = False
    base_model.generation_config.use_cache = False
    patch_nemotron_moe_dtype(base_model, torch)

    lora_config = LoraConfig(
        task_type=adapter_config.get("task_type", "CAUSAL_LM"),
        inference_mode=True,
        r=int(adapter_config["r"]),
        lora_alpha=int(adapter_config["lora_alpha"]),
        lora_dropout=float(adapter_config.get("lora_dropout", 0.0)),
        target_modules=adapter_config["target_modules"],
        bias=adapter_config.get("bias", "none"),
    )
    model = get_peft_model(base_model, lora_config)
    load_adapter_weights_directly(model, adapter_dir)
    model.eval()
    return model, tokenizer, torch


def generate_answers(model, tokenizer, torch_module, rows: pd.DataFrame, run: BackfillRun) -> pd.DataFrame:
    records = []
    device = next(model.parameters()).device
    row_list = list(rows.itertuples(index=False))
    batch_size = max(1, int(GENERATION_BATCH_SIZE))
    for start_index in range(0, len(row_list), batch_size):
        batch = row_list[start_index : start_index + batch_size]
        prompts = [build_prompt(row.prompt) for row in batch]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding=True,
        ).to(device)
        prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
        start = time.time()
        with torch_module.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=run.max_new_tokens,
                do_sample=False,
                use_cache=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        seconds_per_row = (time.time() - start) / max(1, len(batch))
        for offset, row in enumerate(batch):
            generated_ids = output_ids[offset, int(prompt_lengths[offset]) :]
            raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
            records.append(
                {
                    "phase": run.label,
                    "step": run.step,
                    "id": row.id,
                    "family": row.family,
                    "gold": row.answer,
                    "answer": extract_final_answer(raw),
                    "seconds": seconds_per_row,
                    "generated_tokens": int(generated_ids.numel()),
                    "hit_max_new_tokens": int(generated_ids.numel()) >= run.max_new_tokens,
                    "raw_output": raw,
                }
            )
        done = min(start_index + len(batch), len(row_list))
        if done % 16 == 0 or done == len(row_list):
            print(f"generated {done}/{len(row_list)}")

    scored = pd.DataFrame(records)
    scored["match"] = scored["gold"].map(normalize_answer) == scored["answer"].map(normalize_answer)
    scored["empty_answer"] = scored["answer"].astype(str).str.len().eq(0)
    return scored


def generate_probe(model, tokenizer, torch_module, rows: pd.DataFrame, run: BackfillRun) -> pd.DataFrame:
    scored = generate_answers(model, tokenizer, torch_module, rows, run)
    scored["phase"] = run.label
    cols = [
        "phase",
        "step",
        "id",
        "family",
        "answer",
        "seconds",
        "generated_tokens",
        "hit_max_new_tokens",
        "raw_output",
        "gold",
        "match",
    ]
    return scored[[col for col in cols if col in scored.columns]]


def summarize(scored: pd.DataFrame, run: BackfillRun) -> pd.DataFrame:
    rows = []
    for family, part in [("all", scored)] + list(scored.groupby("family")):
        rows.append(
            {
                "checkpoint": run.label,
                "step": run.step,
                "family": family,
                "rows": int(len(part)),
                "matches": int(part["match"].sum()),
                "misses": int((~part["match"]).sum()),
                "accuracy": float(part["match"].mean()) if len(part) else 0.0,
                "empty_answers": int(part["empty_answer"].sum()),
                "hit_max_new_tokens": int(part["hit_max_new_tokens"].sum()),
                "avg_generated_tokens": float(part["generated_tokens"].mean()) if len(part) else 0.0,
                "avg_seconds": float(part["seconds"].mean()) if len(part) else 0.0,
            }
        )
    return pd.DataFrame(rows)


def run_backfill(run: BackfillRun, eval_rows: pd.DataFrame, probe_rows: pd.DataFrame) -> None:
    run.output_dir.mkdir(parents=True, exist_ok=True)
    adapter_dir = extract_adapter(run)
    model, tokenizer, torch_module = load_model(adapter_dir)
    scored = generate_answers(model, tokenizer, torch_module, eval_rows, run)
    summary = summarize(scored, run)
    probe = generate_probe(model, tokenizer, torch_module, probe_rows, run)

    predictions_path = run.output_dir / "generated_eval_predictions.csv"
    summary_path = run.output_dir / "generated_eval_summary.csv"
    probe_path = run.output_dir / "probe_evolution.csv"
    scored.to_csv(predictions_path, index=False)
    summary.to_csv(summary_path, index=False)
    probe.to_csv(probe_path, index=False)
    print("wrote:", predictions_path)
    print("wrote:", summary_path)
    print("wrote:", probe_path)
    print(summary)
    del model, tokenizer
    import gc

    gc.collect()
    torch_module.cuda.empty_cache()


def main() -> None:
    print("Put input zips here:", BACKFILL_INPUT_DIR)
    BACKFILL_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    BACKFILL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    eval_rows, probe_rows = load_rows()
    for run in BACKFILL_RUNS:
        print("\n=== backfill", run.label, "===")
        run_backfill(run, eval_rows, probe_rows)


main()
